In [ ]:
import os, json
from google.colab import drive
drive.mount('/content/drive')

PHASE3_DIR = "/content/drive/MyDrive/PACL_Project/phase3_outputs"
PHASE2_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs"
PHASE4_DIR = "/content/drive/MyDrive/PACL_Project/phase4_outputs"
os.makedirs(PHASE4_DIR, exist_ok=True)

TARGET_SCHOOL = "هندی"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

test_open_records = load_jsonl(f"{PHASE3_DIR}/test_open.jsonl")
print("کلِ رکوردهای test_open:", len(test_open_records))

missing_school = sum(1 for r in test_open_records if r.get("school") is None)
print("رکوردهای بدونِ فیلدِ school:", missing_school)
assert missing_school < len(test_open_records), (
    "❌ هیچ رکوردی فیلدِ school ندارد — قبل از ادامه، منبعِ داده یا نامِ فیلد را بررسی کنید."
)

hendi_indices = [i for i, r in enumerate(test_open_records) if r.get("school") == TARGET_SCHOOL]
hendi_poets = sorted(set(test_open_records[i]["poet"] for i in hendi_indices))
print(f"تعدادِ رکوردهای سبکِ هندی در test_open: {len(hendi_indices)}")
print(f"تعدادِ شاعرانِ یکتایِ سبکِ هندی (zero-shot): {len(hendi_poets)}")
print("شاعران:", hendi_poets)

from collections import Counter
poet_counts = Counter(test_open_records[i]["poet"] for i in hendi_indices)
for p, c in poet_counts.most_common():
    print(f"  {p}: {c} نمونه")

import random
random.seed(42)
MAX_PER_POET = 500
by_poet = {}
for i in hendi_indices:
    by_poet.setdefault(test_open_records[i]["poet"], []).append(i)
sampled_indices = []
for poet, idxs in by_poet.items():
    sampled_indices.extend(idxs if len(idxs) <= MAX_PER_POET else sorted(random.sample(idxs, MAX_PER_POET)))
hendi_indices = sorted(sampled_indices)
sampled_counts = Counter(test_open_records[i]["poet"] for i in hendi_indices)
print(f"\nپس از سقف‌گذاریِ حداکثر {MAX_PER_POET} نمونه به‌ازایِ هر شاعر (برای سرعت و خوانایی نمودار):")
print(f"تعدادِ نهاییِ رکوردهای نمونه‌گیری‌شده: {len(hendi_indices)}")
for p, c in sampled_counts.most_common():
    print(f"  {p}: {c} نمونه")

assert len(hendi_indices) > 0, (
    "❌ هیچ رکوردِ سبکِ هندی‌ای در test_open پیدا نشد — قبل از ادامه بررسی کنید."
)


In [ ]:
import numpy as np

pacl_test_open_emb = np.load(f"{PHASE3_DIR}/test_open_embeddings_byt5base.npy")
print("شکلِ embeddingِ PACL برای کلِ test_open:", pacl_test_open_emb.shape)
assert pacl_test_open_emb.shape[0] == len(test_open_records), (
    "❌ عدمِ تطابقِ تعداد embedding با رکورد — ترتیبِ فایل عوض شده، قبل از ادامه بررسی شود."
)

pacl_hendi_emb = pacl_test_open_emb[hendi_indices]
print("شکلِ embeddingِ PACL برای زیرمجموعهِ سبکِ هندی:", pacl_hendi_emb.shape)


In [ ]:

from transformers import AutoModel, AutoTokenizer
import torch
import time

DRIVE_MODEL_DIR = f"{PHASE2_DIR}/parsbert_baseline3/final"
device = "cuda" if torch.cuda.is_available() else "cpu"

embed_tokenizer = AutoTokenizer.from_pretrained(DRIVE_MODEL_DIR)
embed_model = AutoModel.from_pretrained(DRIVE_MODEL_DIR, output_hidden_states=True).to(device)
embed_model.eval()

EMBED_MAX_LENGTH = 512
EMBED_BATCH_SIZE = 128

@torch.no_grad()
def extract_embeddings(texts, batch_size=EMBED_BATCH_SIZE, max_length=EMBED_MAX_LENGTH):
    all_vecs = []
    use_amp = torch.cuda.is_available()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = embed_tokenizer(
            batch_texts, truncation=True, max_length=max_length,
            padding=True, return_tensors="pt",
        ).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = embed_model(**enc)
        penultimate = out.hidden_states[-2]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        summed = (penultimate.float() * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        mean_pooled = (summed / counts).cpu().numpy()
        all_vecs.append(mean_pooled)
    return np.concatenate(all_vecs, axis=0)

hendi_texts = [test_open_records[i]["text"] for i in hendi_indices]

t0 = time.time()
parsbert_hendi_emb = extract_embeddings(hendi_texts)
print(f"شکلِ embeddingِ ParsBERT (Baseline 3) برایِ زیرمجموعه‌یِ سبکِ هندی: {parsbert_hendi_emb.shape}  ({time.time()-t0:.1f}s)")

assert np.isnan(parsbert_hendi_emb).any(axis=1).sum() == 0, "❌ NaN در embeddingِ ParsBERT!"

np.save(f"{PHASE4_DIR}/parsbert_baseline3_test_open_hendi_embeddings.npy", parsbert_hendi_emb)
print(f"✅ ذخیره شد در {PHASE4_DIR}/parsbert_baseline3_test_open_hendi_embeddings.npy")


In [ ]:
from collections import Counter
from sklearn.metrics import silhouette_score

hendi_poet_labels = [test_open_records[i]["poet"] for i in hendi_indices]

label_counts = Counter(hendi_poet_labels)
valid_idx = [i for i, p in enumerate(hendi_poet_labels) if label_counts[p] >= 2]
valid_labels = [hendi_poet_labels[i] for i in valid_idx]
if len(set(valid_labels)) >= 2:
    sil_pacl = silhouette_score(pacl_hendi_emb[valid_idx], valid_labels)
    sil_parsbert = silhouette_score(parsbert_hendi_emb[valid_idx], valid_labels)
    print("Silhouette Score (بر اساسِ برچسبِ شاعر، فضای اصلی نه t-SNE):")
    print(f"  PACL:      {sil_pacl:.4f}")
    print(f"  ParsBERT:  {sil_parsbert:.4f}")
    print("→ عددِ بالاتر یعنی جداییِ بهترِ خوشه‌های شاعرانِ سبکِ هندی در فضای embedding.")
else:
    print("⚠️ تعدادِ شاعرانِ معتبر (۲+نمونه) کم است — Silhouette محاسبه نمی‌شود.")

from sklearn.manifold import TSNE

n_samples = len(hendi_indices)
perplexity = max(5, min(30, (n_samples - 1) // 3))
print(f"n_samples={n_samples} | perplexity={perplexity}")

tsne_pacl = TSNE(n_components=2, perplexity=perplexity, random_state=42, init="pca", learning_rate="auto")
pacl_2d = tsne_pacl.fit_transform(pacl_hendi_emb)

tsne_parsbert = TSNE(n_components=2, perplexity=perplexity, random_state=42, init="pca", learning_rate="auto")
parsbert_2d = tsne_parsbert.fit_transform(parsbert_hendi_emb)

print("PACL 2D shape:", pacl_2d.shape, "| ParsBERT 2D shape:", parsbert_2d.shape)


In [ ]:
import matplotlib.pyplot as plt

POET_FINGLISH = {
    "اسیر شهرستانی": "Asir Shahrestani",
    "بابافغانی شیرازی": "Baba Feghani Shirazi",
    "رضی‌الدین آرتیمانی": "Razi al-Din Artimani",
    "سلیم تهرانی": "Salim Tehrani",
    "میرداماد": "Mirdamad",
    "ناصر بخارایی": "Naser Bokharaei",
    "وحیدالزمان قزوینی": "Vahid al-Zaman Qazvini",
}

def to_finglish(name, idx):
    return POET_FINGLISH.get(name, f"Poet {idx + 1}")

unique_poets = sorted(set(hendi_poet_labels))
unique_poets_finglish = [to_finglish(p, i) for i, p in enumerate(unique_poets)]
print("نگاشت (فارسی → فینگلیشی برای نمودار):")
for p, fg in zip(unique_poets, unique_poets_finglish):
    print(f"  {p}  ->  {fg}")

cmap = plt.get_cmap("tab20" if len(unique_poets) <= 20 else "nipy_spectral")
poet_to_color = {p: cmap(i / max(len(unique_poets) - 1, 1)) for i, p in enumerate(unique_poets)}
colors = [poet_to_color[p] for p in hendi_poet_labels]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].scatter(pacl_2d[:, 0], pacl_2d[:, 1], c=colors, s=12, alpha=0.7)
axes[0].set_title(f"PACL (epoch9) — Open-set, Sabk-e Hendi (Silhouette={sil_pacl:.3f})" if "sil_pacl" in dir() else "PACL (epoch9) — Open-set, Sabk-e Hendi")

axes[1].scatter(parsbert_2d[:, 0], parsbert_2d[:, 1], c=colors, s=12, alpha=0.7)
axes[1].set_title(f"ParsBERT Baseline 3 — Open-set, Sabk-e Hendi (Silhouette={sil_parsbert:.3f})" if "sil_parsbert" in dir() else "ParsBERT Baseline 3 — Open-set, Sabk-e Hendi")

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

if len(unique_poets) <= 15:
    handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=poet_to_color[p], markersize=8, label=fg) for p, fg in zip(unique_poets, unique_poets_finglish)]
    fig.legend(handles=handles, loc="lower center", ncol=min(len(unique_poets), 5), bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
OUT_PNG = f"{PHASE4_DIR}/tsne_pacl_vs_parsbert_sabk_hendi.png"
plt.savefig(OUT_PNG, dpi=150, bbox_inches="tight")
print(f"✅ نمودار ذخیره شد در {OUT_PNG}")
plt.show()


In [ ]:
import json as _json

summary = {
    "target_school": TARGET_SCHOOL,
    "n_records": len(hendi_indices),
    "n_unique_poets": len(unique_poets),
    "poet_counts": dict(poet_counts),
    "silhouette_pacl": float(sil_pacl) if "sil_pacl" in dir() else None,
    "silhouette_parsbert": float(sil_parsbert) if "sil_parsbert" in dir() else None,
}
with open(f"{PHASE4_DIR}/tsne_sabk_hendi_summary.json", "w", encoding="utf-8") as f:
    _json.dump(summary, f, ensure_ascii=False, indent=2)

print("=== خلاصهِ گامِ ۲ (t-SNE, سبکِ هندی، Open-set) ===")
import pprint
pprint.pprint(summary)
print(f"\n✅ متن/خلاصه ذخیره شد در {PHASE4_DIR}/tsne_sabk_hendi_summary.json")
